In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
import glob
import re

In [2]:
%load_ext autoreload

In [3]:
#Run this to reload the python file
%autoreload 2
from utils import *

# Harmonization

### Importing Data

In [4]:
# local file path
path = './Data/IMN_raw/IMN.xlsx'

**Get sheet names**

In [5]:
file_info = sheet_list(path)

**Reading the info's sheet from the IMN's original file**

In [6]:
# read one sheet into a DataFrame
meta = pd.read_excel(path, sheet_name=file_info.get('info_sheet'))

In [7]:
metadata = info(meta)

In [5]:
# csv with short term station data
st_df = pd.read_csv('./Data/metadata/Short_Term_IMN.csv')

### Coordinates transformation

The original coordinates of the IMN are in `Degrees Minutes Seconds` and the desired format is `Decimal Degrees`

In [7]:
decimal_degrees_lat = []

for coord in st_df['Latitud Norte']:
    transf = dms_to_dd(coord)
    decimal_degrees_lat.append(transf)

In [10]:
decimal_degrees_lon = []

for coord in metadata['Longitud Oeste']:
    transf = dms_to_dd(coord)
    transf = -transf
    decimal_degrees_lon.append(transf)

In [10]:
# Add columns for lat and lon in decimal degrees
metadata['lat'] = decimal_degrees_lat
metadata['lon'] = decimal_degrees_lon

In [11]:
metadata.to_csv('./Data/metadata/IMN_stations.csv')

### Sheet format changes

In [12]:
aws_1 = pd.read_excel(path, sheet_name=file_info.get('list_names')[0])

This is the original sheet format of each AWS from IMN, lets to change it to a standard one, where:
- there are no empty rows at the beginning
- columns with corresponding names
- time change from `01:00-23:00` to `00:00-24:00`
- time change from local time to UTC time
    - Costa Rica has one time zone, which is located in the UTC−06:00 zone, 6 hours behind Coordinated Universal Time (UTC)
- save each AWS sheet as an independet file

In [ ]:
%%time

for x in file_info.get('list_names'):
    print(f'Working in {x}')
    try:
        # read one sheet into a DataFrame
        station = pd.read_excel(path, sheet_name=str(x))
        
        # remove empty rows at start of Dataframe
        df = preprocess(station)
    
        # change the format
        df = formating(df)
        
        aws_number = x.replace(' ', '')
    
        # Create DataFrames for pcp data
        df_pcp = pd.DataFrame({'station_number': aws_number, 'date': df['Date'], 'pcp': df['pcp']})
        
        # convert the date column to datetime format
        df_pcp['date'] = pd.to_datetime(df_pcp['date'], infer_datetime_format=True)

        # time change to UTC time
        df_pcp['date'] = df_pcp['date'] + timedelta(hours=6)
    
        # Save the data to separate CSV files
        df_pcp.to_csv(f'./Data/IMN_raw/{aws_number}_IMN.csv', index=False)
    
        print(f'csv created for {x}')
    except Exception as e:
        print(f'Error processing {x}: {str(e)}')
        continue

# Quallity Control

### Importing data

In [5]:
# path file
paths = glob.glob('./Data/IMN_raw/*IMN*csv')

In [15]:
start_date = '2001-01-01 00:00:00'
end_date = '2022-12-31 23:00:00'

In [16]:
percentage = []

for path in paths:
    
    df = pd.read_csv(path)
    
    df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
    
    # checking missing dates
    m_dates = missing_dates(df, 'date', start_date, end_date, 'H')
    
    # create a DataFrame with missing dates and 'NA' in the 'pcp' column
    m_dates = pd.DataFrame({'station_number': path[15:-8], 'date': m_dates, 'pcp': np.nan})
    
    df = pd.concat([df, m_dates], ignore_index=True)
    
    df['date'] = pd.to_datetime(df['date'])  
    
    df = df.sort_values(by='date')
    
    # replace missing values to nan
    df = missing_values(df, 'pcp', -9)
    
    #calculate percentage of missing data in a specific time range
    perc = nan_percentage(df, 'date', 'pcp', start_date, end_date, 'H')
    percentage.append(perc)
    
    if perc < 10:
        df.to_csv(f"./Data/harmonized/{path[15:-8]}_imn.csv")

In [17]:
numbers = []
for path in paths:
    number = path[15:-8]
    numbers.append(number)

In [18]:
tmp = pd.DataFrame()
tmp['station_number'] = numbers
tmp['percentage'] = percentage

In [19]:
# adding the percentage values into the metadata file
meta = pd.read_csv('./Data/metadata/IMN_stations.csv')

In [20]:
meta['Número'] = meta['Número'].astype(str)
merged_df = meta.merge(tmp, left_on='Número', right_on='station_number')

In [21]:
merged_df.to_csv('./Data/metadata/IMN_stations_rev.csv')

### Join all files into one

In [31]:
# path file
paths = glob.glob('./Data/harmonized/*imn*ST*csv')

In [32]:
df_comb = pd.DataFrame()

In [33]:
df_comb['date'] = pd.date_range(start=start_date, end=end_date, freq='H')

In [34]:
for path in paths:
    df = pd.read_csv(path)
    
    #df_comb[path[18:-11]] = df['pcp']
    df_comb[path[18:-11]] = df['pcp']

In [35]:
# Columns to convert from str to numeric
columns_to_convert = ['72153', '69679', '69713', '69723', '69681', '72163', '76063', '69739', '74051', '74063', '69647', '76065', 
                    '72183', '69699', '69633', '72189', '69677', '69701', '76055', '74053', '72149']

df_comb[columns_to_convert] = df_comb[columns_to_convert].replace(',', '.', regex=True)

# Convert the specified columns to numeric type
df_comb[columns_to_convert] = df_comb[columns_to_convert].apply(pd.to_numeric, errors='coerce')

In [37]:
df_comb.to_csv('./Data/harmonized/unif_IMN_ST.csv', index=False)

### Nate

In [22]:
# path file
paths = glob.glob('./Data/IMN_raw/*IMN_ST*csv')

In [23]:
# Nate UTC time
start_date = '2017-10-04 00:00:00'
end_date = '2017-10-06 23:00:00'

In [24]:
for path in paths:
    
    df = pd.read_csv(path)
    
    df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
    
    # checking missing dates
    m_dates = missing_dates(df, 'date', start_date, end_date, 'H')
    
    # create a DataFrame with missing dates and 'NA' in the 'pcp' column
    m_dates = pd.DataFrame({'station_number': path[15:-11], 'date': m_dates, 'pcp': np.nan})
    
    df = pd.concat([df, m_dates], ignore_index=True)
    
    df['date'] = pd.to_datetime(df['date'])  
    
    df = df.sort_values(by='date')
    
    # replace missing values to nan
    df = missing_values(df, 'pcp', -9)
    
    df.to_csv(f"./Data/harmonized/imn_ST/{path[15:-11]}_imn_ST.csv")